# Models supervised

## Import Module

In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import cross_val_score

## MLPClassifier

#### On va d'abord essayer de faire une version classique avec ce que l'on connait

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladie
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen

clf = MLPClassifier(random_state=1, max_iter=300).fit(X_train, y_train) # Créer et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive a classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Créer la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteint de stone sont atteint de stone.

raise SystemExit(0)

In [ ]:
df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

scaler = MinMaxScaler()
matrices = scaler.fit_transform(matrices)

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladie
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # permet de garder des proportions coherentes entre les maladies (evite de n'avoir aucune image d'un type ou inversement)
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen

clf = MLPClassifier(random_state=1, max_iter=500, hidden_layer_sizes=(256, 128, 64)).fit(X_train, y_train) # Créer et entraîne le modèle

print("Test 5 examens")
scores = cross_val_score(clf, X, y, cv=5)
print("Score des 5 examens :", scores)
print("Precision moyenne :", scores.mean())

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive a classer )

print("Accuracy :", accuracy) # Print la précision

probabilites = clf.predict_proba(X_test)
print(probabilites[:5])

y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Créer la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteint de stone sont atteint de stone.


Test 5 examens


on obtient :
```

Calcul des 5 examens (patientez quelques minutes)...
Scores des 5 examens : [1.         1.         1.         1.         0.99959823]
Précision moyenne : 0.9999196464443552
Accuracy : 1.0
[[9.99970927e-01 5.49619114e-08 1.07761648e-06 2.79401663e-05]
 [8.58051446e-08 9.99962803e-01 3.70486470e-05 6.20944912e-08]
 [9.99999029e-01 5.82697481e-07 2.97454818e-07 9.09543532e-08]
 [1.54482835e-09 9.99988598e-01 1.13995222e-05 6.78334239e-10]
 [9.45347915e-06 5.18671088e-05 9.99926281e-01 1.23980405e-05]]
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]

```

#### Explication

Pour utiliser MLPClassifier, nous avons essayé plusieurs approches différentes.
Tout d'abord, nous avons divisé (split) nos données pour entraîner le modèle sur 80 % et le tester sur les 20 % restants.
Ensuite, nous avons pris nos échantillons et nous les avons entraînés avec plus ou moins de max_iter.
Et pour mesurer la précision de notre modèle, nous affichons le score d'Accuracy et la matrice de confusion.

Après le premier test, nous obtenons un score d'Accuracy de 1 et une matrice complètement faussée.

Après avoir analysé la situation, on se rend compte d'un problème potentiel qui peut en être la cause : nous n'avons pas les mêmes proportions de sujets sains, de cailloux (calculs), de tumeurs et de kystes. Quand nous prenons 80 % des données pour l'entraînement, il se peut que les proportions ne soient pas respectées.
Imaginons qu'au total on ait 1000 images de cailloux mélangées parmi les 12 000 : rien ne nous garantit que le modèle va tirer des cailloux et qu'il va en prendre assez.
Pour cela, nous allons ajouter stratify=y dans le train_test_split, ce qui garantit d'avoir une proportion minimale respectée.

Cela a permis d'obtenir un meilleur score, mais la matrice de confusion était encore faussée :
```
Accuracy : 0.5671593830334191
[[621   0 306   0]
 [  0 800 470   0]
 [  0   0 344   0]
 [  0   0 571   0]]
```
Pourquoi ?
Car on voit que nous avons quasiment toutes les prédictions concentrées dans 1 seul choix. C'est théoriquement ce que l'on veut, mais notre modèle est trop parfait. Il y a 2 possibilités : la première, on a une fuite de données (data leak) et il apprend par cœur les réponses ; l'option 2, le modèle est surchargé en valeurs avec les arrays.

Nous allons donc procéder dans l'ordre pour essayer de régler ces problèmes. On va d'abord regarder au niveau des arrays.
MLPClassifier n'aime pas les grands nombres, et comme les arrays d'images ont des valeurs comprises entre 0 et 255, on va réduire cela avec MinMaxScaler pour normaliser nos données entre 0 et 1.

Comme cela ne règle pas notre problème, on va essayer de configurer les hidden_layer_sizes.
Comme on a énormément de données en entrée et qu'on doit avoir un petit résultat à la sortie (4 classes), cela crée un gros goulot d'étranglement. La normalisation aide beaucoup, mais les hidden_layer_sizes vont permettre de mettre X neurones au début, puis de faire travailler Y neurones sur le résultat des neurones X, ce qui facilite grandement l'apprentissage.

Mais même malgré cela, nous obtenons des matrices parfaites et des scores parfaits comme :
```
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]
```
Nous avons donc testé une dernière analyse en exécutant cross_val_score. Cela va permettre de vraiment savoir si nous avons un coup de chance au tirage ou si le modèle est simplement très performant.
On obtient :
```
Test 5 examens
Score des 5 examens : [1.         1.         1.         1.         0.99959823]
Precision moyenne : 0.9999196464443552
Accuracy : 1.0

[[9.99970927e-01 5.49619114e-08 1.07761648e-06 2.79401663e-05]
 [8.58051446e-08 9.99962803e-01 3.70486470e-05 6.20944912e-08]
 [9.99999029e-01 5.82697481e-07 2.97454818e-07 9.09543532e-08]
 [1.54482835e-09 9.99988598e-01 1.13995222e-05 6.78334239e-10]
 [9.45347915e-06 5.18671088e-05 9.99926281e-01 1.23980405e-05]]
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]
```
On a donc notre réponse : nos pourcentages de certitude sont de l'ordre de 99,996 % et, comme nous faisons un cross_val_score sur 5 examens tournants, on sait de manière mathématique que ce n'est pas dû à de la malchance ou au hasard du tirage. Le modèle exploite bien une faille dans les données.


Pour regler ce probleme on va le regler a la source, on va utiliser un systeme de groupe (voir preprocessing)



In [3]:
df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices


X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladie
y = df["target"] # Donne les différentes maladies possibles
groupes = df['groupe']

best_accuracy = 0
meilleur_split = 0

for i in range (1, 5):
    gss = GroupShuffleSplit(n_splits=i, test_size=0.2, random_state=1)
    train_idx, test_idx = next(gss.split(X, y, groups=groupes))
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)   # fit SEULEMENT sur train
    X_test = scaler.transform(X_test)
    
    clf = MLPClassifier(random_state=1, max_iter=300, hidden_layer_sizes=(256, 128, 64)).fit(X_train, y_train) # Créer et entraîne le modèle
    
    
    accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive a classer )
    
    print("Accuracy :", accuracy) # Print la précision
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        meilleur_split = i

print(f"la meilleur accuracy est avec : {meilleur_split}, pour un score de {best_accuracy}")

y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Créer la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteint de stone sont atteint de stone.


Accuracy : 0.9251585623678646


/home/gabin/Documents/Project/B-SPE-311-LIL-3-1-cvrie-2/Myvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Accuracy : 0.9302325581395349
Accuracy : 0.9251585623678646


/home/gabin/Documents/Project/B-SPE-311-LIL-3-1-cvrie-2/Myvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Accuracy : 0.9408033826638478
la meilleur accuracy est avec : 4, pour un score de 0.9408033826638478
[[818   0   0   0]
 [  0 914 129   0]
 [  0  11 214   0]
 [  0   0   0 279]]


#### On obtient alors 
```
Accuracy : 0.9251585623678646
Accuracy : 0.9302325581395349
Accuracy : 0.9251585623678646
Accuracy : 0.9408033826638478
la meilleur accuracy est avec : 4, pour un score de 0.9408033826638478
[[818   0   0   0]
 [  0 914 129   0]
 [  0  11 214   0]
 [  0   0   0 279]]
```

Ce resultat est deja beaucoup plus satisfaisant que le precedent.

##### Actuellement nous avons une version complete mais nous avencon a l'aveugle, il faut pour avoir une precision coherente et proche de la realite essayer plusieurs parametre de `max_iter` et `hidden_layer_sizes`. Le `n_splits` est mis sur 5 car c'est la norme la plus part des models utilisent un decoupage en 5.

##### Pour ce faire on ne va pas utiliser des boucles qui changent les valeurs, car notre model commence a etre assez lourd pour gerer plusieurs combinaison de parametres. Nous allons utiliser `GridSearchCV`, il ne reduira pas forcement le temps d'execution mais va repartir la charge sur tout les coeurs/ threads du proceseurs et alleger la charge. Il nous donnera parfaitement le meilleur resultat avec nos parametres.

In [ ]:
df = pd.read_csv("../../Supervise.csv") 
matrices = np.load('../../Matrice.npy', allow_pickle=True)

# classique data
X = matrices 
y = df["target"]
groupes = df['groupe']

# on creer une pipeline qui va contenir notre MinMaxSclaer ainsi que le MLPClaffifier
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),                 
    ('mlp', MLPClassifier(random_state=1))      
])

# No sparametres a tester pou le model
param_grid = {
    'mlp__hidden_,layer_sizes': [(128, 64), (256, 128 64)],
    'mlp__max_iter': [300, 500]
}

# le melange des groupe tout en faisant attention aux meme patients pour le data leak
gss = GroupShuffleSplit(n_splits=3, test_size=0.2, random_state=1)


# utilisation de GridSearchCV pour aller chercher les meilleurs parametres dans notre model
grid_search = GridSearchCV(
    estimator=pipeline,    # chaine de montage a realiser
    param_grid=param_grid,    # parametres qu'on va tester
    cv=gss,    # methode pour melanger 
    n_jobs=-1     # nombre de coeur physique
)

# on entraine le model pou chercher le meilleur
grid_search.fit(X, y, groups=groupes)

# on affiche nos deux meilleur parametre
print("Meilleurs parametres trouves: ", grid_search.best_params_)


##### Comment ca marche ?
Concretement on va utiliser un entrainement pour trouver les meilleurs parametres avec une pipeline.
On va definir nos methodes dans la pipeline (Normalisation avec MinMaxScaler et notre model pour lui trouver les meilleurs parametres)

On defini ensuite les parametres a tester ainsi que nos donnes qui vont servir pour l'entrainement et le test (dans notre cas on coupe en 5 mais on utilise `GroupShuffleSplit` pour ne pas avoir de dataleak entre entrainement et test)

On defini apres notre `GridSearchCV`, on lui donne les instructions (ceux de la pipeline) puis les parametres a tester, puis comment on va melanger puis le nombre de coeur (-1 utilisant tout les coeurs physique disponible)

ON peut ensuite entrainer et afficher les meilleurs parametres avec `best_params_`.


Informations utile :
    - POur acceder et utiliser une sous methode de notre pipeline il faut mettre ca cle suivit de `__` par exemple si la cle est `test` omn doit mettre `test__`


Apres execution avec 
```
param_grid = {
    'mlp__hidden_,layer_sizes': [(128, 64), (256, 128 64)],
    'mlp__max_iter': [300, 500]
}
```

On obtient : `Meilleurs parametres trouves:  {'mlphidden_layer_sizes': (128, 64), 'mlpmax_iter': 300}`.

Nous avons alors notre reponse, nos meilleurs parametres sont en couche cache 128 et 64 et en iteration 300.

Ceci conclus notre MLPClassifier, nous avons donc un score proche des 99 %, allant de 92 a 99 selon les parametres (malheuresement les limitation machine ne nous permettent pas d'obtenir un resultat 100% correcte.

In [ ]:
### 